# NB04 — Sensitivity Analysis (Extended)
**Benin Least-Cost Electrification Analysis**

Tests how technology selection and programme cost respond to key uncertain parameters.

## New parameters added in this version
The dual-rate demand model introduces two additional sensitivity parameters:

| Parameter | Base | Range | Expected impact |
|---|---|---|---|
| `demand_intensity_growth` | 3%/yr | 2–4% | Affects LCOE denominators, MG threshold timing |
| `population_growth_rate` | 2.7%/yr | 2.0–3.3% | Affects new HH count, pop-growth CAPEX |

Combined, these 7 parameters (up from 5) give a more complete picture of model uncertainty.

## Total scenarios: 21 (up from 17)


In [ ]:
import sys, os
from pathlib import Path

project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ast
import copy
import warnings

from src.config import GENERAL, DEMAND, SHS as SHS_CONFIG, DISCOUNT_RATE
from src.costs.grid_extension  import add_grid_lcoe
from src.costs.mini_grid       import add_minigrid_lcoe, MG_SUBTYPES
from src.costs.shs             import add_shs_lcoe
from src.least_cost.technology_selector import select_least_cost, technology_summary
from src.demand.demand_estimator import add_demand_columns, shs_price_at_year
import src.costs.mini_grid        as mg_mod
import src.costs.shs              as shs_mod
import src.demand.demand_estimator as dem_mod


print('Imports ✓')
print(f'  Base intensity growth : {DEMAND["demand_intensity_growth_rate"]*100:.1f}%/yr')
print(f'  Base pop growth (eff) : {DEMAND["population_growth_rate"]*DEMAND["rural_unelec_share"]*100:.2f}%/yr')
combined = ((1+DEMAND['demand_intensity_growth_rate'])*(1+DEMAND['population_growth_rate']*DEMAND['rural_unelec_share'])-1)*100
print(f'  Combined growth       : {combined:.2f}%/yr')


## 1. Load Base Results

Loads two files:
1. **`settlements_demand_*.geojson`** (from NB02) — the input to re-run LCOE for each scenario
2. **`settlements_lcoe_*.geojson`** (from NB03) — the base-case results for comparison

Loading the demand file (not the LCOE file) as the scenario input ensures each scenario
starts from a clean state — it recomputes all LCOEs from scratch with overridden parameters,
avoiding any contamination from the previous scenario.

**If either file is missing:** run NB02 and NB03 first.


In [ ]:
from pathlib import Path

PROCESSED_DIR = Path('..') / 'data' / 'processed'
candidates = sorted(PROCESSED_DIR.glob('settlements_lcoe_*.geojson'), reverse=True)

if candidates:
    print(f'✓ Found {len(candidates)} settlements_lcoe file(s):')
    for f in candidates:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name}  ({size_mb:.1f} MB)')
    print(f'\nWill use: {candidates[0].name}')
else:
    print('✗ No settlements_lcoe_*.geojson found in data/processed/')
    print('  Run NB03 cell 25 (Save Results) first, then re-run this notebook.')
    raise FileNotFoundError('Run NB03 save cell first')


In [ ]:
import geopandas as gpd
import ast
from pathlib import Path

PROCESSED_DIR = Path('..') / 'data' / 'processed'
candidates = sorted(PROCESSED_DIR.glob('settlements_lcoe_*.geojson'), reverse=True)
if not candidates:
    raise FileNotFoundError('No settlements_lcoe_*.geojson — run NB03 first')

BASE_PATH = candidates[0]
print(f'Loading: {BASE_PATH.name}')
gdf_base = gpd.read_file(BASE_PATH)
print(f'Loaded {len(gdf_base):,} settlements | {len(gdf_base.columns)} columns')

# Restore demand_timeseries from string
if 'demand_timeseries' in gdf_base.columns:
    if isinstance(gdf_base['demand_timeseries'].iloc[0], str):
        gdf_base['demand_timeseries'] = gdf_base['demand_timeseries'].apply(ast.literal_eval)
    print('demand_timeseries restored ✓')
else:
    # Reconstruct from demand_year0_kwh with 4% annual growth
    print('demand_timeseries missing — reconstructing from demand_year0_kwh')
    T = 15
    gdf_base['demand_timeseries'] = gdf_base['demand_year0_kwh'].apply(
        lambda d0: [float(d0) * (1.04**t) for t in range(T+1)]
    )
    print('demand_timeseries reconstructed ✓')

# Column check
key_cols = ['elec_status','demand_year0_kwh','demand_timeseries',
            'dist_nearest_electrified_km','GridDistKm','num_connections',
            'lcoe_grid','lcoe_mg_solar','lcoe_shs','least_cost_tech']
print('\nKey column check:')
for col in key_cols:
    status = '✓' if col in gdf_base.columns else '✗ MISSING'
    print(f'  {col:<35}: {status}')

# Base case summary
print('\n=== BASE CASE RESULTS ===')
for tech, n in gdf_base['least_cost_tech'].value_counts().items():
    print(f'  {tech:<35}: {n:,} ({n/len(gdf_base)*100:.1f}%)')

TECH_COLORS = {
    'Already Electrified'            : '#BDBDBD',
    'Grid Extension'                 : '#1565C0',
    'Mini-Grid: Solar PV Only'       : '#F9A825',
    'Mini-Grid: Solar-Diesel Hybrid' : '#E53935',
    'Mini-Grid: Mini-Hydro'          : '#00897B',
    'SHS'                            : '#43A047',
}
UNELEC_TECHS = ['Grid Extension','Mini-Grid: Solar PV Only',
                'Mini-Grid: Solar-Diesel Hybrid','Mini-Grid: Mini-Hydro','SHS']


## 2. Scenario Runner — `run_scenario()`

The `run_scenario()` function re-runs the full LCOE pipeline for a single parameter combination.

**How it works:**
1. Takes the base-case demand DataFrame as input (always the same starting point)
2. Overrides specific parameters in `MG_SUBTYPES` and/or `SHS_PARAMS` dictionaries **in-memory**
   using `copy.deepcopy()` to avoid mutating the shared module state
3. Calls `add_minigrid_lcoe()`, `add_shs_lcoe()`, and `select_least_cost()` with the overridden values
4. Returns a `technology_summary` dict: `{technology_name: pct_of_settlements}`
5. Restores the original parameters before returning

**Why deepcopy?** Python module-level dictionaries are shared across all calls.
Without deepcopy, overriding `MG_SUBTYPES['mg_solar']['min_demand_kwh']` in scenario 1
would persist into scenario 2. Deepcopy creates an isolated working copy.

**Runtime:** ~3–4 seconds per scenario × 17 scenarios ≈ ~1 minute total.


In [ ]:
import importlib

def run_scenario(
    gdf_base,           # settlements WITH demand columns (from NB02)
    scenario_name,

    # ── Mini-grid parameters ───────────────────────────────────────────
    battery_replace_yr       = 10,
    battery_cost_kwh         = 270,
    mg_solar_conn_cost       = 280,
    mg_solar_min_demand      = 15_000,
    mg_hybrid_conn_cost      = 300,
    mg_hydro_conn_cost       = 350,
    mg_hybrid_min_demand     = 25_000,
    mg_hydro_min_demand      = 10_000,
    diesel_price_override    = None,

    # ── SHS parameters ─────────────────────────────────────────────────
    shs_t2_cost              = None,    # None = use price-decline curve default
    shs_price_decline_rate   = 0.05,    # 5%/yr decline

    # ── Demand growth parameters ────────────────────────────────────────
    demand_intensity_rate    = None,    # None = use config default (0.03)
    population_growth_rate   = None,    # None = use config default (0.027)

    dist_col                 = 'dist_nearest_electrified_km',
    exclude_hydro            = False,
    exclude_hybrid           = False,
):
    """
    Re-runs the full LCOE pipeline with overridden parameters.
    Returns technology share dict: {tech_name: pct_of_unelec_settlements}
    """
    import src.costs.mini_grid as mg_mod
    import src.costs.shs as shs_mod
    import src.demand.demand_estimator as dem_mod
    from src.config import DEMAND as DEMAND_CFG, SHS as SHS_CFG
    import copy

    # ── 1. Deep-copy module-level dicts to avoid cross-contamination ──────
    orig_mg       = copy.deepcopy(mg_mod.MG_SUBTYPES)
    orig_shs_t2   = SHS_CFG['tier_2']['capex_per_unit']
    orig_int_rate = DEMAND_CFG['demand_intensity_growth_rate']
    orig_pop_rate = DEMAND_CFG['population_growth_rate']

    try:
        # ── 2. Override MG parameters ──────────────────────────────────────
        mg_mod.MG_SUBTYPES['mg_solar']['battery_replace_yr'] = battery_replace_yr
        mg_mod.MG_SUBTYPES['mg_solar']['battery_cost_kwh']   = battery_cost_kwh
        mg_mod.MG_SUBTYPES['mg_solar']['capex_per_conn']     = mg_solar_conn_cost
        mg_mod.MG_SUBTYPES['mg_solar']['min_demand_kwh']     = mg_solar_min_demand
        mg_mod.MG_SUBTYPES['mg_hybrid']['capex_per_conn']    = mg_hybrid_conn_cost
        mg_mod.MG_SUBTYPES['mg_hydro']['capex_per_conn']     = mg_hydro_conn_cost
        mg_mod.MG_SUBTYPES['mg_hybrid']['min_demand_kwh']    = mg_hybrid_min_demand
        mg_mod.MG_SUBTYPES['mg_hydro']['min_demand_kwh']     = mg_hydro_min_demand
        if exclude_hydro:
            mg_mod.MG_SUBTYPES['mg_hydro']['min_demand_kwh']  = 999_999_999
        if exclude_hybrid:
            mg_mod.MG_SUBTYPES['mg_hybrid']['min_demand_kwh'] = 999_999_999

        # ── 3. Override SHS price ──────────────────────────────────────────
        if shs_t2_cost is not None:
            SHS_CFG['tier_2']['capex_per_unit'] = shs_t2_cost

        # ── 4. Override demand growth rates — re-run demand model if changed
        demand_changed = (demand_intensity_rate is not None or
                          population_growth_rate is not None)
        if demand_changed:
            if demand_intensity_rate is not None:
                DEMAND_CFG['demand_intensity_growth_rate'] = demand_intensity_rate
            if population_growth_rate is not None:
                DEMAND_CFG['population_growth_rate'] = population_growth_rate
            # Re-run demand estimation with new rates
            tmp = add_demand_columns(gdf_base.copy())
        else:
            tmp = gdf_base.copy()

        # ── 5. Re-run LCOE pipeline ────────────────────────────────────────
        unelec_mask = tmp['elec_status'] == 'unelectrified'
        tmp_unelec  = tmp[unelec_mask].copy()

        # Grid LCOE
        tmp_unelec = add_grid_lcoe(tmp_unelec, dist_col=dist_col)

        # Mini-grid LCOE
        if diesel_price_override is not None:
            tmp_unelec['DieselPrice'] = diesel_price_override
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            tmp_unelec = add_minigrid_lcoe(tmp_unelec)

        # SHS LCOE
        tmp_unelec = add_shs_lcoe(tmp_unelec)

        # Merge back and select
        for col in tmp_unelec.columns:
            if col not in tmp.columns:
                tmp.loc[unelec_mask, col] = tmp_unelec[col].values
            elif col in ['lcoe_grid','lcoe_mg_solar','lcoe_mg_hybrid',
                         'lcoe_mg_hydro','lcoe_shs',
                         'lcoe_minigrid','minigrid_subtype']:
                tmp.loc[unelec_mask, col] = tmp_unelec[col].values

        tmp = select_least_cost(tmp)

        # ── 6. Return technology shares (unelectrified only) ──────────────
        unelec_out = tmp[tmp['elec_status'] == 'unelectrified']
        shares = (unelec_out['least_cost_tech']
                  .value_counts(normalize=True) * 100
                  ).round(2).to_dict()
        shares['_scenario'] = scenario_name
        shares['_hh_total'] = float(unelec_out['num_households'].sum())
        shares['_hh_yT']    = float(unelec_out['num_households_yT'].sum()) \
                              if 'num_households_yT' in unelec_out.columns else None
        return shares, tmp

    finally:
        # ── 7. Always restore original parameters ─────────────────────────
        mg_mod.MG_SUBTYPES = orig_mg
        SHS_CFG['tier_2']['capex_per_unit']         = orig_shs_t2
        DEMAND_CFG['demand_intensity_growth_rate']  = orig_int_rate
        DEMAND_CFG['population_growth_rate']        = orig_pop_rate

print('run_scenario() defined ✓  (covers: battery, connection cost, SHS price,')
print('                           diesel price, MG demand threshold,')
print('                           demand intensity rate, population growth rate)')

def shares_to_df(results_list, UNELEC_TECHS=None):
    """Convert list of share-dicts from run_scenario to a long-format DataFrame."""
    rows = []
    for d in results_list:
        scenario = d.get('_scenario', '')
        for tech, pct in d.items():
            if tech.startswith('_'):
                continue
            rows.append({'Scenario': scenario, 'Technology': tech, 'Pct_settlements': pct})
    return pd.DataFrame(rows)


## 3. Battery Replacement Sensitivity

**What we are testing:** Whether the choice of battery chemistry (LFP vs lead-acid) meaningfully
changes which technology wins for each settlement.

**Base case:** LFP battery at $270/kWh with replacement at year 10.

**Scenarios:**
- Lead-acid (year 5, $150/kWh): older, cheaper, shorter-lived — replace twice in 15-year horizon
- LFP base case (year 10, $270/kWh): current model assumption
- LFP optimistic (year 12, $270/kWh): well-managed LFP systems exceeding standard lifetime

**Expected finding:** The LCOE spread between lead-acid and LFP is approximately **$0.07/kWh**
on solar MG. This is not large enough to flip any settlement's technology selection —
SHS still wins the small settlements regardless of battery cost, and MG still wins the large ones.

**Significance:** Confirms that battery procurement decisions affect project finance and tariffs,
but do **not** change the planning-level technology map.


In [ ]:
battery_results = []
for yr, label in [(5,'Lead-acid (yr 5)'),(8,'LFP conservative (yr 8)'),
                   (10,'LFP base case (yr 10)'),(12,'LFP optimistic (yr 12)')]:
    s, _ = run_scenario(gdf_base, label, battery_replace_yr=yr)
    battery_results.append(s)
    print(f'  {label} done')

battery_df = shares_to_df(battery_results)
print('\n=== BATTERY SENSITIVITY ===')
pivot = battery_df.pivot_table(index='Scenario', columns='Technology',
                                values='Pct_settlements', fill_value=0)
print(pivot[[t for t in UNELEC_TECHS if t in pivot.columns]].round(1).to_string())


## 4. Mini-Grid Connection Cost Sensitivity

**What we are testing:** How the LV connection cost per household ($150–$350/HH) affects
the Solar MG vs SHS decision boundary.

**Base case:** $280/HH (ABERME 2022 average).

**Scenarios:**
- $150/HH: donor-subsidised connection (e.g. ABERME rural electrification programme subsidy)
- $280/HH: commercial/base-case
- $350/HH: full cost-recovery commercial price

**Why this matters:** Connection cost is a fixed per-household charge — it has a larger
proportional impact on small settlements (few HH, high cost per kWh) than on large ones.
Lowering connection cost to $150 can make Solar MG viable for settlements just below the
15,000 kWh/yr demand threshold.

**Expected finding:** ±1.2 percentage points on Solar MG share — moderate impact but not critical.


In [ ]:
conn_results = []
for cost, label in [(150,'Subsidised ($150)'),(200,'Low ($200)'),
                     (280,'Base case ($280)'),(350,'Full cost ($350)')]:
    s, _ = run_scenario(gdf_base, label, mg_solar_conn_cost=cost,
                        mg_hybrid_conn_cost=cost+20, mg_hydro_conn_cost=cost+70)
    conn_results.append(s)
    print(f'  {label} done')

conn_df = shares_to_df(conn_results)
print('\n=== CONNECTION COST SENSITIVITY ===')
pivot = conn_df.pivot_table(index='Scenario', columns='Technology',
                             values='Pct_settlements', fill_value=0)
print(pivot[[t for t in UNELEC_TECHS if t in pivot.columns]].round(1).to_string())


## 5. SHS Kit Cost Sensitivity

**What we are testing:** How declining SHS market prices affect the SHS vs Solar MG boundary.

**Base case:** $150/kit (GOGLA West Africa Q3 2023 average).

**Scenarios:**
- $120/kit: projected 2030 price (consistent with BloombergNEF clean energy price decline curves)
- $150/kit: current base case
- $180/kit: conservative/high-cost scenario

**Why this matters:** A lower SHS kit price reduces SHS LCOE, making SHS more competitive
against Solar MG for settlements near the 15,000 kWh/yr threshold.

**Expected finding:** Same ±1.2 pp swing as connection cost — symmetric effect.
If both SHS cost falls AND connection cost rises simultaneously, the combined effect could
reach ±3–4 pp (multiplicative, not additive). This interaction is not tested here but should
be noted as a limitation.


In [ ]:
shs_results = []
for cost, label in [(120,'Low kit ($120)'),(150,'Base case ($150)'),
                     (180,'High kit ($180)')]:
    s, _ = run_scenario(gdf_base, label, shs_t2_cost=cost)
    shs_results.append(s)
    print(f'  {label} done')

shs_df = shares_to_df(shs_results)
print('\n=== SHS KIT COST SENSITIVITY ===')
pivot = shs_df.pivot_table(index='Scenario', columns='Technology',
                            values='Pct_settlements', fill_value=0)
print(pivot[[t for t in UNELEC_TECHS if t in pivot.columns]].round(1).to_string())


## 6. Diesel Price Sensitivity

**What we are testing:** Whether diesel price variation affects the Solar MG vs Hybrid MG
technology selection.

**Base case:** $0.85/L (SBEE regulated price, spatially varying in the full model).

**Scenarios:**
- $0.75/L: low scenario (subsidised price, possible policy change)
- $0.85/L: current regulated price
- $1.00/L: high scenario (unsubsidised + transport markup in remote areas)

**Why this has negligible impact in practice:**
Hybrid MG wins **zero settlements** in the base case — Solar MG already outcompetes
it in every feasible settlement. Diesel price variation only affects the Hybrid LCOE
(via the `annual_fuel_cost` term in `mg_lcoe()`), but since Hybrid never wins,
the sensitivity is trivially zero.

**Expected finding:** 0.1 pp swing — negligible. Confirms that the Hybrid MG
technology is not competitive in Benin under current assumptions.


In [ ]:
diesel_results = []
for price, label in [(0.75,'Low ($0.75/L)'),(0.85,'Base ($0.85/L)'),
                      (1.00,'High ($1.00/L)')]:
    s, _ = run_scenario(gdf_base, label, diesel_price_override=price)
    diesel_results.append(s)
    print(f'  {label} done')

diesel_df = shares_to_df(diesel_results)
print('\n=== DIESEL PRICE SENSITIVITY ===')
pivot = diesel_df.pivot_table(index='Scenario', columns='Technology',
                               values='Pct_settlements', fill_value=0)
print(pivot[[t for t in UNELEC_TECHS if t in pivot.columns]].round(1).to_string())


## 7. MG Demand Threshold Sensitivity — THE CRITICAL PARAMETER

**What we are testing:** Whether the ESMAP minimum demand threshold for mini-grid feasibility
(15,000 kWh/yr for Solar MG) is the right choice — and how sensitive results are to it.

**Base case:** 15,000 kWh/yr (ESMAP 2019: minimum viable 3–5 kW solar MG system).

**Scenarios:**
- 10,000 kWh/yr: lower threshold — more settlements qualify for MG → Solar MG share rises
- 15,000 kWh/yr: base case
- 25,000 kWh/yr: higher threshold — fewer settlements qualify → Solar MG share falls

**Why this is the critical parameter:**
The demand threshold directly controls the *size of the mini-grid market*.
Every settlement with demand between the low and high threshold is on the fence —
changing the threshold literally moves them from SHS to Solar MG or back.

**Expected finding:** ±5.0 pp swing (3.6% to 8.6% Solar MG share).
This is 4× larger than any other parameter tested — it is the single most important
assumption in the model and should be validated with field data before investment decisions.


In [ ]:
thresh_results = []
for solar_t, label in [
    (10_000, 'Low threshold (10,000 kWh)'),
    (15_000, 'Base case (15,000 kWh)'),
    (25_000, 'High threshold (25,000 kWh)'),
]:
    s, _ = run_scenario(gdf_base, label,
                        mg_solar_min_demand=solar_t,
                        mg_hybrid_min_demand=int(solar_t*1.67),
                        mg_hydro_min_demand=int(solar_t*0.67))
    thresh_results.append(s)
    print(f'  {label} done')

thresh_df = shares_to_df(thresh_results)
print('\n=== DEMAND THRESHOLD SENSITIVITY ===')
pivot = thresh_df.pivot_table(index='Scenario', columns='Technology',
                               values='Pct_settlements', fill_value=0)
print(pivot[[t for t in UNELEC_TECHS if t in pivot.columns]].round(1).to_string())


## 8. Demand Intensity Growth Rate Sensitivity

Tests the impact of the income-driven per-HH electricity demand growth rate (base: 3%/yr).
This directly affects mini-grid viability by changing how quickly settlements cross the 15,000 kWh/yr threshold.

- **Low (2%)**: slow income growth — post-shock scenario, conservative
- **Base (3%)**: GDP ×0.75 income elasticity — central estimate
- **High (4%)**: GDP-matched growth — optimistic / original assumption

In [ ]:
intensity_results = []
for rate, label in [(0.02,'Low intensity (2%)'),(0.03,'Base (3%)'),(0.04,'High (4%)')]:
    r, _ = run_scenario(gdf_base, label, demand_intensity_rate=rate)
    intensity_results.append(r)
    mg_share = r.get('Mini-Grid: Solar PV Only', 0)
    shs_share = r.get('SHS', 0)
    print(f'{label:<30}: Solar MG={mg_share:.1f}%  SHS={shs_share:.1f}%')


## 9. Population Growth Rate Sensitivity

Tests the impact of Benin's population growth rate on new household formation in unelectrified areas.

- **Low (2.0%)**: optimistic fertility decline scenario
- **Base (2.7%)**: UN WPP 2024 medium-fertility variant
- **High (3.3%)**: UN WPP 2024 high-fertility variant

This does not change technology *selection* (the same technology wins for existing HH),
but it changes **total CAPEX** and the number of connections needed by 2040.

In [ ]:
pop_results = []
for rate, label in [(0.020,'Low pop (2.0%)'),(0.027,'Base (2.7%)'),(0.033,'High (3.3%)')]:
    r, _ = run_scenario(gdf_base, label, population_growth_rate=rate)
    pop_results.append(r)
    mg_share  = r.get('Mini-Grid: Solar PV Only', 0)
    shs_share = r.get('SHS', 0)
    hh_yT     = r.get('_hh_yT', 0) or 0
    print(f'{label:<28}: Solar MG={mg_share:.1f}%  SHS={shs_share:.1f}%  ')
          # HH count at year 15 varies by pop growth


## 8b. Technology Exclusion Scenarios — Hydro Off → Solar/Hybrid

Tests what happens when mini-hydro is **excluded from the technology mix**.
This simulates:
- A policy decision to not develop hydro (environmental concerns, data uncertainty)
- A risk-adjusted scenario where HydroSHEDS data quality is too poor to rely on

**What to expect:**
- Hydro settlements (328 sites, 1.9%) redistribute to Solar MG or Hybrid MG
- Solar MG share rises by approximately 1.5–2.0 percentage points
- SHS share is unchanged (hydro sites have sufficient demand for MG)
- Total investment changes as Solar MG replaces cheaper hydro


In [ ]:
# ── No-Hydro scenario: hydro sites reassigned to solar or hybrid ──────────────
print('Running technology exclusion scenarios...')

# Scenario A: Hydro excluded — sites go to solar MG or hybrid
s_no_hydro, gdf_no_hydro = run_scenario(
    gdf_base, 'No hydro (→ Solar/Hybrid)',
    exclude_hydro=True
)

# Scenario B: Both hydro and hybrid excluded — sites go to solar MG only
s_solar_only, gdf_solar_only = run_scenario(
    gdf_base, 'Solar MG only (no hydro, no hybrid)',
    exclude_hydro=True,
    exclude_hybrid=True
)

# Base case for comparison
s_base, _ = run_scenario(gdf_base, 'Base case (all technologies)')

exclusion_df = shares_to_df([s_base, s_no_hydro, s_solar_only])

print('\n=== TECHNOLOGY EXCLUSION RESULTS ===')
pivot_ex = exclusion_df.pivot_table(
    index='Scenario', columns='Technology',
    values='Pct_settlements', fill_value=0
)
print(pivot_ex[[t for t in UNELEC_TECHS if t in pivot_ex.columns]].round(1).to_string())

print('\n=== WHAT HAPPENS TO HYDRO SITES ===')
hydro_base   = (gdf_base['least_cost_tech'] == 'Mini-Grid: Mini-Hydro').sum()
hydro_no_hyd = (gdf_no_hydro['least_cost_tech'] == 'Mini-Grid: Mini-Hydro').sum()
print(f'  Hydro sites (base case)      : {hydro_base:,}')
print(f'  Hydro sites (hydro excluded) : {hydro_no_hyd:,}  (should be 0)')
print()
print('  Redistribution (hydro excluded):')
for tech in UNELEC_TECHS:
    n_base = (gdf_base.loc[gdf_base['least_cost_tech']=='Mini-Grid: Mini-Hydro',
               'least_cost_tech'] == tech).sum()
    # For these same settlements in the no-hydro scenario
    hydro_site_ids = gdf_base[gdf_base['least_cost_tech']=='Mini-Grid: Mini-Hydro'].index
    if len(hydro_site_ids) > 0 and tech in gdf_no_hydro.columns or True:
        n_new = (gdf_no_hydro.loc[
            gdf_no_hydro.index.isin(hydro_site_ids), 'least_cost_tech'
        ] == tech).sum()
        if n_new > 0:
            print(f'    → {tech}: {n_new:,} sites')

print()
print('=== INVESTMENT IMPACT ===')
for scen_df, label in [(gdf_base, 'Base case'),
                        (gdf_no_hydro, 'No hydro'),
                        (gdf_solar_only, 'Solar only')]:
    total = scen_df['least_cost_capex'].sum() / 1e6
    print(f'  {label:<35}: ${total:.1f}M')


## 8. Tornado Chart — Parameter Impact Summary

Visualises the sensitivity results as a **tornado chart**: horizontal bars sorted by
the width of the sensitivity range (swing = max − min).

**Reading the chart:**
- Each bar represents one parameter
- Bar width = the total swing in Solar MG share (pp = percentage points)
- Bars extending LEFT of the baseline: lower parameter value → fewer MG settlements
- Bars extending RIGHT of the baseline: higher parameter value → more MG settlements

**Expected order (widest to narrowest):**
1. Demand threshold (5.0 pp) — CRITICAL
2. Connection cost (1.2 pp) — Moderate
3. SHS kit cost (1.2 pp) — Moderate
4. Diesel price (0.1 pp) — Negligible
5. Battery cycle (0.1 pp) — Negligible

**Key takeaway for the report:**
SHS dominates in ALL 17 scenarios (42–49% of settlements).
Even if the demand threshold is halved to 10,000 kWh/yr, SHS still wins for
the majority of Benin's unelectrified settlements.


In [ ]:
# ── Tornado chart — all 7 parameters ─────────────────────────────────────────
# Collect min/max Solar MG share across each parameter's scenarios
all_results = {
    'Demand threshold':        thresh_results,
    'Connection cost':         conn_results,
    'SHS kit cost':            shs_results,
    'Diesel price':            diesel_results,
    'Battery cycle':           battery_results,
    'Demand intensity rate':   intensity_results,
    'Population growth rate':  pop_results,
}

BASE_TECH = 'Mini-Grid: Solar PV Only'
base_run, _ = run_scenario(gdf_base, 'base')
base_val  = base_run.get(BASE_TECH, 0)

print(f'Base case Solar MG share: {base_val:.1f}%')
print()

tornado_data = []
for param, results in all_results.items():
    vals = [r.get(BASE_TECH, 0) for r in results]
    if vals:
        lo, hi = min(vals), max(vals)
        swing = hi - lo
        tornado_data.append({'param': param, 'lo': lo, 'hi': hi, 'swing': swing})

tornado_data.sort(key=lambda x: x['swing'], reverse=True)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_facecolor('#FAFAFA')

colors = {'critical': '#C62828', 'moderate': '#E65100', 'minor': '#2E7D32'}
for i, d in enumerate(tornado_data):
    y    = len(tornado_data) - 1 - i
    base = base_val
    lo_w = base - d['lo']
    hi_w = d['hi'] - base
    color = colors['critical'] if d['swing'] >= 4 else colors['moderate'] if d['swing'] >= 1 else colors['minor']
    if lo_w > 0:
        ax.barh(y, -lo_w, left=base, height=0.55, color=color, alpha=0.75)
    if hi_w > 0:
        ax.barh(y,  hi_w, left=base, height=0.55, color=color, alpha=0.75)
    ax.text(-0.3, y, d['param'], ha='right', va='center', fontsize=10)
    ax.text(d['hi'] + 0.15, y, f'{d["swing"]:.1f}pp', ha='left', va='center', fontsize=9, fontweight='bold')

ax.axvline(base_val, color='#1565C0', lw=1.8, ls='--', label=f'Base: {base_val:.1f}%')
ax.set_xlabel('Solar MG share (% of unelectrified settlements)', fontsize=11)
ax.set_title('Tornado Chart — Solar MG Share Sensitivity\n(7 parameters, base case = {:.1f}%)'.format(base_val), fontsize=12)
ax.set_yticks([])
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../data/outputs/tables/tornado_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('=== SENSITIVITY SUMMARY ===')
print(f'{"Parameter":<30} {"Low":>6} {"Base":>6} {"High":>6} {"Swing":>7} {"Impact"}')
print('-' * 70)
for d in tornado_data:
    impact = 'Critical' if d['swing']>=4 else 'Moderate' if d['swing']>=1 else 'Minor'
    print(f'{d["param"]:<30} {d["lo"]:>6.1f} {base_val:>6.1f} {d["hi"]:>6.1f} {d["swing"]:>7.1f}pp  {impact}')

# Convert to DataFrame for saving in cell 30
tornado_df = pd.DataFrame([
    {'Parameter': d['param'], 'Min': d['lo'], 'Base': base_val,
     'Max': d['hi'], 'Range': d['swing'],
     'Impact': 'Critical' if d['swing']>=4 else 'Moderate' if d['swing']>=1 else 'Minor'}
    for d in tornado_data
])
print(f'\ntornado_df built: {len(tornado_df)} parameters ✓')


## 8. Population Growth Rate Sensitivity

**What we are testing:** How uncertainty in Benin's population growth rate
(UN range: 2.0%–3.3%/yr depending on fertility scenario) affects the
total number of new households needing connection by 2040.

**This does not change technology selection** — new HH in a settlement
get the same technology as existing HH. It changes the total CAPEX and
the population-growth component of the programme cost.

**Expected finding:** ±15–20% swing on population-growth CAPEX (~±$13M),
modest effect on total programme cost (~±1%).


In [ ]:
# Population growth rate sensitivity
import importlib, copy
import src.demand.demand_estimator as de_mod

pop_results = []
for pop_rate, label in [
    (0.020, 'Low UN variant (2.0%/yr)'),
    (0.027, 'Base case (2.7%/yr)'),
    (0.033, 'High UN variant (3.3%/yr)'),
]:
    # Override in-memory
    orig_pop  = de_mod.DEMAND.get('population_growth_rate', 0.027)
    de_mod.DEMAND['population_growth_rate'] = pop_rate

    tmp = gdf_base.copy()
    tmp = de_mod.add_demand_columns(tmp)

    unelec_tmp = tmp[tmp['elec_status']=='unelectrified']
    new_hh     = unelec_tmp.get('hh_growth_total', 0)
    if hasattr(new_hh, 'sum'): new_hh = new_hh.sum()

    pop_results.append({
        'label': label,
        'pop_growth_rate': f"{pop_rate*100:.1f}%",
        'new_hh_2040': round(new_hh),
        'pop_growth_capex_m': round(new_hh * 200 / 1e6, 1),
    })
    de_mod.DEMAND['population_growth_rate'] = orig_pop  # restore

import pandas as pd
df_pop = pd.DataFrame(pop_results)
print("=== POPULATION GROWTH SENSITIVITY ===")
print(df_pop.to_string(index=False))


## 9. Demand Intensity Growth Sensitivity

**What we are testing:** Whether the 3%/yr income-driven per-HH demand
growth is the right rate — and how it affects LCOE values and the mini-grid
demand threshold.

The key mechanism: higher intensity growth → settlements reach the 15,000 kWh/yr
mini-grid threshold earlier → more settlements qualify for MG over the horizon.

In [ ]:
# Demand intensity growth sensitivity
intensity_results = []
for rate, label in [
    (0.02, 'Low intensity (2%/yr)'),
    (0.03, 'Base case (3%/yr)'),
    (0.04, 'High intensity (4%/yr)'),
]:
    summary, _ = run_scenario(
        gdf_base,
        scenario_name=label,
        demand_intensity_rate=rate,
    )
    summary['label'] = label
    summary['intensity_rate'] = f"{rate*100:.0f}%"
    intensity_results.append(summary)

df_int = pd.DataFrame(intensity_results)
print("=== DEMAND INTENSITY GROWTH SENSITIVITY ===")
cols = ['label','intensity_rate','SHS','Mini-Grid: Solar PV Only',
        'Mini-Grid: Mini-Hydro','Grid Extension']
cols_present = [c for c in cols if c in df_int.columns]
print(df_int[cols_present].to_string(index=False))


## 9. Save Results

Saves the sensitivity analysis output to `data/outputs/tables/sensitivity_results_{ts}.csv`.

**File format:** Each row is a scenario. Columns are technology names.
Values are the percentage share of unelectrified settlements assigned to each technology.

This file is not currently used by the Streamlit dashboard but can be loaded directly
for report-writing or further analysis.

**Also printed to console:** A summary table comparing all scenarios to the base case,
highlighting which parameters move results the most — ready to copy into a report.


In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
TABLE_DIR = Path('..') / 'data' / 'outputs' / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)

all_results = pd.concat([
    battery_df, conn_df, shs_df, diesel_df, thresh_df, exclusion_df
], ignore_index=True)
all_results.to_csv(TABLE_DIR / f'sensitivity_results_{ts}.csv', index=False)
print(f'Saved: sensitivity_results_{ts}.csv')

if 'tornado_df' in dir() and len(tornado_df) > 0:
    tornado_df.to_csv(TABLE_DIR / f'tornado_summary_{ts}.csv', index=False)
    print(f'Saved: tornado_summary_{ts}.csv')
    print('\n=== TORNADO SUMMARY ===')
    print(tornado_df[['Parameter','Min','Max','Range']]
          .sort_values('Range', ascending=False).to_string(index=False))
else:
    print('Tornado chart skipped — no Solar MG variation across scenarios')

print('\n=== ALL SENSITIVITY RESULTS ===')
pivot = all_results.pivot_table(
    index='Scenario', columns='Technology',
    values='Pct_settlements', fill_value=0
)
print(pivot[[t for t in UNELEC_TECHS if t in pivot.columns]].round(1).to_string())
